# Data augmentation experiments
This notebook applies SMOTE, ADASYN and a simple GAN approach to augment the minority class and retrain models.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

url = 'https://storage.googleapis.com/qwasar-public/track-ds/creditcard.csv'
df = pd.read_csv(url)
X = df.drop('Class', axis=1)
y = df['Class']
scaler = StandardScaler()
X[['Time','Amount']] = scaler.fit_transform(X[['Time','Amount']])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)


In [ ]:
# SMOTE and ADASYN
# Augmentation experiments: scaler fit on train only, LR + SMOTE added, and metrics reported
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE, ADASYN
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix


# Load cleaned data
df = pd.read_csv('./creditcard_full.csv')
X = df.drop('Class', axis=1)
y = df['Class']

# Split before scaling/augmentation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Fit scaler on training only
scaler = StandardScaler()
X_train[['Time','Amount']] = scaler.fit_transform(X_train[['Time','Amount']])
X_test[['Time','Amount']] = scaler.transform(X_test[['Time','Amount']])

# --- SMOTE augmentation (on training data only) ---
sm = SMOTE(random_state=42)
X_sm, y_sm = sm.fit_resample(X_train, y_train)
print("After SMOTE: X_sm shape:", X_sm.shape, "Positive count:", int(y_sm.sum()))

# Logistic Regression baseline and LR + SMOTE
lr_orig = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_orig.fit(X_train, y_train)
preds_orig = lr_orig.predict(X_test)
probs_orig = lr_orig.predict_proba(X_test)[:,1]

lr_sm = LogisticRegression(max_iter=1000, random_state=42)
lr_sm.fit(X_sm, y_sm)
preds_sm = lr_sm.predict(X_test)
probs_sm = lr_sm.predict_proba(X_test)[:,1]

def summarize(name, y_true, preds, probs):
    auc = roc_auc_score(y_true, probs)
    prec = precision_score(y_true, preds)
    rec = recall_score(y_true, preds)
    f1 = f1_score(y_true, preds)
    cm = confusion_matrix(y_true, preds)
    print(f"\n{name} -> AUC: {auc:.4f}  Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}")
    print("Confusion Matrix:\n", cm)

summarize("LR - Original", y_test, preds_orig, probs_orig)
summarize("LR - SMOTE", y_test, preds_sm, probs_sm)

# Add RF on SMOTE for reference
rf_sm = RandomForestClassifier(n_estimators=100, random_state=42)
rf_sm.fit(X_sm, y_sm)
preds_rf_sm = rf_sm.predict(X_test)
probs_rf_sm = rf_sm.predict_proba(X_test)[:,1]
summarize("RF - SMOTE", y_test, preds_rf_sm, probs_rf_sm)




In [ ]:
# Simple GAN for tabular data (toy example using keras)
# Note: This is a minimal GAN for feature generation; for production use consider CTGAN or Tabular GANs
from tensorflow import keras
from tensorflow.keras import layers

# Prepare minority samples
min_idx = (y_train==1)
X_min = X_train[min_idx]

latent_dim = 16
# Generator
gen = keras.Sequential([layers.Input(shape=(latent_dim,)), layers.Dense(64, activation='relu'), layers.Dense(X_min.shape[1], activation='linear')])
# Discriminator
disc = keras.Sequential([layers.Input(shape=(X_min.shape[1],)), layers.Dense(64, activation='relu'), layers.Dense(1, activation='sigmoid')])

gan_input = layers.Input(shape=(latent_dim,))
generated = gen(gan_input)
disc.trainable = False
validity = disc(generated)

gan = keras.Model(gan_input, validity)

gan.compile(loss='binary_crossentropy', optimizer='adam')
disc.compile(loss='binary_crossentropy',optimizer='adam')

# Training loop (very small epochs for demo) - this will take time on large data
import numpy as np
batch_size = 128
steps = 1000
for step in range(steps):
    # Train discriminator
    idx = np.random.randint(0, X_min.shape[0], batch_size)
    real = X_min.values[idx]
    noise = np.random.normal(0,1,(batch_size, latent_dim))
    fake = gen.predict(noise)
    d_loss_real = disc.train_on_batch(real, np.ones((batch_size,1)))
    d_loss_fake = disc.train_on_batch(fake, np.zeros((batch_size,1)))
    # Train generator
    g_loss = gan.train_on_batch(noise, np.ones((batch_size,1)))
    if step % 200 == 0:
        print('step', step, 'd_loss_real', d_loss_real, 'd_loss_fake', d_loss_fake, 'g_loss', g_loss)

# Generate synthetic minority samples
noise = np.random.normal(0,1,(len(X_min), latent_dim))
syn = gen.predict(noise)

# Combine and train RF
X_gan = pd.concat([X_train, pd.DataFrame(syn, columns=X_train.columns)], ignore_index=True)
y_gan = pd.concat([y_train, pd.Series([1]*len(syn))], ignore_index=True)

rf_gan = RandomForestClassifier(n_estimators=100, random_state=42)
rf_gan.fit(X_gan, y_gan)
probs_gan = rf_gan.predict_proba(X_test)[:,1]
print('GAN RF AUC:', roc_auc_score(y_test, probs_gan))


In [ ]:
# Save augmentation models/results
import joblib
joblib.dump({'lr_orig': lr_orig, 'lr_sm': lr_sm, 'rf_sm': rf_sm}, 'augmented_models.pkl')
print("\nSaved augmented_models.pkl")
